In [1]:
import os
import time
from collections import defaultdict
from typing import Dict, List, Tuple
import numpy as np
from tqdm import tqdm
from numpy.matlib import zeros


# Import the methods and metrics
from tools.ewca import EWCA
from tools.sedmtg import SEDMTG, ProteinNetwork
from tools.mpcc import MPCC
from tools.metrics import compute_metrics

class ProteinComplexExtractor:
    def __init__(self):
        self.protein_to_id: Dict[str, int] = {}
        self.id_to_protein: Dict[int, str] = {}
        self.known_complexes: List[List[str]] = []  # Store known complexes as lists of protein names
    
    def load_known_complexes(self, filepath: str):
        """Load known complexes from file, handling both formats:
        - With header: complex_id\tproteins (separated by ;)
        - Without header: complex_id\tproteins (separated by spaces)
        """
        self.known_complexes = []
        
        with open(filepath, 'r') as f:
            # Check if file has header
            first_line = f.readline().strip()
            
            # Detect if header exists
            has_header = first_line.lower() in ['complex_id\tproteins', 'complex_id proteins']
            
            # Reset file pointer if no header
            if not has_header:
                f.seek(0)
            
            for line_num, line in enumerate(f, 1 if has_header else 0):
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                    
                parts = line.split('\t')
                if len(parts) < 2:
                    print(f"Line {line_num} ignored - invalid format: {line}")
                    continue
                    
                # Handle both formats:
                if ';' in parts[1]:  # Format with semicolon-separated proteins
                    proteins = parts[1].split(';')
                else:  # Format with space-separated proteins
                    proteins = parts[1].split()
                    
                # Clean proteins (remove empty entries and whitespace)
                proteins = [p.strip() for p in proteins if p.strip()]
                
                if not proteins:
                    print(f"Line {line_num} ignored - no valid proteins: {line}")
                    continue
                    
                self.known_complexes.append(proteins)
    
    def generate_complexes(self, ewca_file: str, sedmtg_file: str, output_file: str, metrics_file: str):
        """
        Generate complexes and calculate metrics for each solution.
        Each solution is compared independently against the known complexes.
        """
        start_time = time.time()
        
        all_complexes = []
        metrics_results = []
        
        # Method 1: EWCA with varying ss_threshold (8 solutions)
        print("\nRunning EWCA method...")
        ss_thresholds = np.linspace(0.4, 0.68, 8)
        
        for i, ss_threshold in enumerate(tqdm(ss_thresholds, desc="EWCA progress")):
            ewca = EWCA(ewca_file, ss_threshold)
            
            # Run EWCA and get complexes (modified to return complexes instead of saving)
            ewca.load_interactions()
            ewca.calculate_jaccard_distance()
            ewca.calculate_ecv2_weights()
            core_complexes = ewca.detect_core_complexes()
            complexes = ewca.find_attachments(core_complexes)
            filtered_complexes = ewca.filter_redundant_complexes(complexes)
            
            # Convert to protein names and filter (keep only complexes with ≥3 proteins)
            solution_complexes = [
                [ewca.id_to_protein[pid] for pid in members]
                for members in filtered_complexes.values()
                if len(members) >= 3
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 1,
                'method': 'EWCA',
                'param': f"ss_threshold={ss_threshold:.2f}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                metrics_result = compute_metrics(solution_complexes, self.known_complexes)
                metrics.update({
                    'PPV': metrics_result["PPV"],
                    'recall': metrics_result["Recall (Sn)"],  # Notez le changement ici
                    'fmeasure': metrics_result["F-mesure"],
                    'coverage_rate': metrics_result["Covered Rate"],
                    'accuracy': metrics_result["Accuracy"],
                    'mmr': metrics_result["MMR"],
                    'jaccard': metrics_result["Jaccard"],
                    'total_score': metrics_result["Score Total"]
                })
                
                print(f"\nEWCA Solution {i+1} (threshold={ss_threshold:.2f}):")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {metrics['fmeasure']:.4f}, Accuracy: {metrics['accuracy']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 1,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        
        # Method 2: SEDMTG (8 solutions)
        print("\nRunning SEDMTG method...")
        network = ProteinNetwork.from_weighted_network(sedmtg_file, has_header=True)
        sedmtg = SEDMTG(network, iterations=5)
        
        for i in tqdm(range(8), desc="SEDMTG progress"):
            # Run SEDMTG (modified to return complexes for each iteration)
            protein_complexes = sedmtg.detect_complexes()
            
            # Convert to list format and filter
            solution_complexes = [
                proteins for proteins in protein_complexes.values()
                if len(proteins) >= 3
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 9,
                'method': 'SEDMTG',
                'param': f"iteration_{i + 1}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                metrics_result = compute_metrics(solution_complexes, self.known_complexes)
                metrics.update({
                    'PPV': metrics_result["PPV"],
                    'recall': metrics_result["Recall (Sn)"],  # Notez le changement ici
                    'fmeasure': metrics_result["F-mesure"],
                    'coverage_rate': metrics_result["Covered Rate"],
                    'accuracy': metrics_result["Accuracy"],
                    'mmr': metrics_result["MMR"],
                    'jaccard': metrics_result["Jaccard"],
                    'total_score': metrics_result["Score Total"]
                })
                
                
                print(f"\nSEDMTG Solution {i+9}:")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {metrics['fmeasure']:.4f}, Accuracy: {metrics['accuracy']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 9,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        # Method 3: MPCC with varying filter thresholds (8 solutions)
        print("\nRunning MPCC method...")
        mpcc = MPCC()
        
        # Load interactions once (same file as EWCA)
        mpcc.load_interactions(ewca_file)
        mpcc.remove_false_positives()
        
        # Calculate topology scores and combined weights (done once)
        N = len(mpcc.id_label)
        topo_weights = mpcc.calculate_topology_scores(mpcc.relations, N)
        combined_weights = zeros((N, N))
        
        # Create weight matrix
        weight_matrix = zeros((N, N))
        for (i,j), w in mpcc.weights.items():
            weight_matrix[i,j] = w
        
        # Combine weights
        for i in range(N):
            for j in range(N):
                if i < j and weight_matrix[i,j] > 0 and topo_weights[i,j] > 0:
                    combined_weights[i,j] = 2 * weight_matrix[i,j] * topo_weights[i,j] / (weight_matrix[i,j] + topo_weights[i,j])
                    combined_weights[j,i] = combined_weights[i,j]
        
        # Detect seeds once (they don't depend on the filter threshold)
        seeds = mpcc.detect_seeds(list(mpcc.relations.keys()), mpcc.relations, combined_weights)
        
        # Vary the filter threshold from 0.1 to 0.8 in 8 steps
        filter_thresholds = np.linspace(0.1, 0.8, 8)
        
        for i, threshold in enumerate(tqdm(filter_thresholds, desc="MPCC progress")):
            # Identify complexes with current filter threshold
            complexes = mpcc.identify_complexes(seeds, mpcc.relations, combined_weights)
            
            # Calculate scores
            complex_scores = {}
            final_complexes = defaultdict(list)
            count = 1
            
            for cid in list(complexes.keys()):
                if len(complexes[cid]) >= 3:
                    final_complexes[count] = complexes[cid]
                    complex_scores[count] = mpcc.graph_entropy(complexes[cid], mpcc.relations, combined_weights)
                    count += 1
            
            # Filter with current threshold
            filtered = mpcc.filter_redundant(final_complexes, threshold=threshold)
            
            # Convert to protein names
            solution_complexes = [
                [mpcc.id_label[pid] for pid in members]
                for members in filtered.values()
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 17,  # Starts after EWCA (8) and SEDMTG (8)
                'method': 'MPCC',
                'param': f"filter_threshold={threshold:.2f}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                metrics_result = compute_metrics(solution_complexes, self.known_complexes)
                metrics.update({
                    'PPV': metrics_result["PPV"],
                    'recall': metrics_result["Recall (Sn)"],  # Notez le changement ici
                    'fmeasure': metrics_result["F-mesure"],
                    'coverage_rate': metrics_result["Covered Rate"],
                    'accuracy': metrics_result["Accuracy"],
                    'mmr': metrics_result["MMR"],
                    'jaccard': metrics_result["Jaccard"],
                    'total_score': metrics_result["Score Total"]
                })
                
                print(f"\nMPCC Solution {i+17} (threshold={threshold:.2f}):")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {metrics['fmeasure']:.4f}, Accuracy: {metrics['accuracy']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 17,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        
        # Save results
        self._save_results(all_complexes, output_file)
        self._save_metrics(metrics_results, metrics_file)
        
        elapsed_time = time.time() - start_time
        print(f"\nTotal processing time: {elapsed_time:.2f} seconds")
        print(f"Complexes saved to {output_file}")
        print(f"Metrics saved to {metrics_file}")
    
    def _save_results(self, all_complexes: List[Dict], output_file: str):
        """Save all complexes to output file in the required format"""
        with open(output_file, 'w') as f:
            f.write("SolutionID\tComplexID\tProteins\n")
            for complex_data in all_complexes:
                f.write(f"{complex_data['solution_id']}\t{complex_data['complex_id']}\t{' '.join(complex_data['proteins'])}\n")
    
    def _save_metrics(self, metrics_results: List[Dict], metrics_file: str):
        """Save metrics to a TSV file with additional information"""
        with open(metrics_file, 'w') as f:
            # Write header
            f.write("SolutionID\tMethod\tParameters\tDetectedComplexes\tKnownComplexes\t"
                    "PPV\tRecall\tF-measure\tCoverageRate\tAccuracy\tMMR\tJaccard\tTotalScore\n"
                )

            # Write data
            for result in metrics_results:
                f.write(
                    f"{result['solution_id']}\t"
                    f"{result['method']}\t"
                    f"{result['param']}\t"
                    f"{result['detected_complexes']}\t"
                    f"{result['known_complexes']}\t"
                    f"{result.get('PPV', 0):.4f}\t"  # Utilisation de get() avec valeur par défaut
                    f"{result.get('recall', 0):.4f}\t"
                    f"{result['fmeasure']:.4f}\t"
                    f"{result['coverage_rate']:.4f}\t"
                    f"{result['accuracy']:.4f}\t"
                    f"{result['mmr']:.4f}\t"
                    f"{result['jaccard']:.4f}\t"
                    f"{result['total_score']:.4f}\n"
                    
                )

def main():
    # Configuration
    ewca_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_levure.txt"
    sedmtg_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/tmp/GO_weighted_BIOGRID_levure.txt"
    known_complexes_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/complexes/BIOGRID_levure.txt"
    output_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/detected_complexes_BIOGRID_levure.txt"
    metrics_file = "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/metrics/metrics_BIOGRID_levure.tsv"
    
    # Create extractor
    extractor = ProteinComplexExtractor()
    
    # Load known complexes if available
    if os.path.exists(known_complexes_file):
        print("Loading known complexes...")
        extractor.load_known_complexes(known_complexes_file)
        print(f"Loaded {len(extractor.known_complexes)} known complexes")
    else:
        print("Warning: No known complexes file found at", known_complexes_file)
    
    # Generate complexes and metrics
    print("\nGenerating protein complexes and calculating metrics...")
    extractor.generate_complexes(
        ewca_file=ewca_file,
        sedmtg_file=sedmtg_file,
        output_file=output_file,
        metrics_file=metrics_file
    )
    
    print("\nProcessing complete!")

if __name__ == "__main__":
    main()

Loading known complexes...
Loaded 246 known complexes

Generating protein complexes and calculating metrics...

Running EWCA method...


EWCA progress:   0%|          | 0/8 [00:00<?, ?it/s]

Total number of proteins: 4235
Total number of interactions: 24465


EWCA progress:  12%|█▎        | 1/8 [00:06<00:48,  6.90s/it]


EWCA Solution 1 (threshold=0.40):
Complexes: 1349, F-measure: 0.4314, Accuracy: 0.5067
Total number of proteins: 4235
Total number of interactions: 24465


...


MPCC Solution 22 (threshold=0.60):
Complexes: 542, F-measure: 0.3987, Accuracy: 0.4689


MPCC progress:  88%|████████▊ | 7/8 [02:23<00:14, 14.64s/it]


MPCC Solution 23 (threshold=0.70):
Complexes: 635, F-measure: 0.4074, Accuracy: 0.4786


MPCC progress: 100%|██████████| 8/8 [02:36<00:00, 19.61s/it]


MPCC Solution 24 (threshold=0.80):
Complexes: 748, F-measure: 0.4179, Accuracy: 0.4879

Total processing time: 4885.22 seconds
Complexes saved to /Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/detected_complexes_BIOGRID_levure.txt
Metrics saved to /Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/metrics/metrics_BIOGRID_levure.tsv

Processing complete!


In [2]:
import os
import time
from collections import defaultdict
from typing import Dict, List, Tuple
import numpy as np
from tqdm import tqdm
from numpy.matlib import zeros

# Import the methods and metrics
from tools.ewca import EWCA
from tools.sedmtg import SEDMTG, ProteinNetwork
from tools.mpcc import MPCC
from tools.metrics import compute_metrics

class ProteinComplexExtractor:
    def __init__(self):
        self.protein_to_id: Dict[str, int] = {}
        self.id_to_protein: Dict[int, str] = {}
        self.known_complexes: List[List[str]] = []  # Store known complexes as lists of protein names
    
    def load_known_complexes(self, filepath: str):
        """Load known complexes from file, handling both formats:
        - With header: complex_id\tproteins (separated by ;)
        - Without header: complex_id\tproteins (separated by spaces)
        """
        self.known_complexes = []
        
        with open(filepath, 'r') as f:
            # Check if file has header
            first_line = f.readline().strip()
            
            # Detect if header exists
            has_header = first_line.lower() in ['complex_id\tproteins', 'complex_id proteins']
            
            # Reset file pointer if no header
            if not has_header:
                f.seek(0)
            
            for line_num, line in enumerate(f, 1 if has_header else 0):
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                    
                parts = line.split('\t')
                if len(parts) < 2:
                    print(f"Line {line_num} ignored - invalid format: {line}")
                    continue
                    
                # Handle both formats:
                if ';' in parts[1]:  # Format with semicolon-separated proteins
                    proteins = parts[1].split(';')
                else:  # Format with space-separated proteins
                    proteins = parts[1].split()
                    
                # Clean proteins (remove empty entries and whitespace)
                proteins = [p.strip() for p in proteins if p.strip()]
                
                if not proteins:
                    print(f"Line {line_num} ignored - no valid proteins: {line}")
                    continue
                    
                self.known_complexes.append(proteins)
    
    def generate_complexes(self, ewca_file: str, sedmtg_file: str, output_file: str, metrics_file: str):
        """
        Generate complexes and calculate metrics for each solution.
        Each solution is compared independently against the known complexes.
        """
        start_time = time.time()
        
        all_complexes = []
        metrics_results = []
        
        # Method 1: EWCA with varying ss_threshold (8 solutions)
        print("\nRunning EWCA method...")
        ss_thresholds = np.linspace(0.72, 0.76, 2)
        
        for i, ss_threshold in enumerate(tqdm(ss_thresholds, desc="EWCA progress")):
            ewca = EWCA(ewca_file, ss_threshold)
            
            # Run EWCA and get complexes (modified to return complexes instead of saving)
            ewca.load_interactions()
            ewca.calculate_jaccard_distance()
            ewca.calculate_ecv2_weights()
            core_complexes = ewca.detect_core_complexes()
            complexes = ewca.find_attachments(core_complexes)
            filtered_complexes = ewca.filter_redundant_complexes(complexes)
            
            # Convert to protein names and filter (keep only complexes with ≥3 proteins)
            solution_complexes = [
                [ewca.id_to_protein[pid] for pid in members]
                for members in filtered_complexes.values()
                if len(members) >= 3
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 1,
                'method': 'EWCA',
                'param': f"ss_threshold={ss_threshold:.2f}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                metrics_result = compute_metrics(solution_complexes, self.known_complexes)
                metrics.update({
                    'PPV': metrics_result["PPV"],
                    'recall': metrics_result["Recall (Sn)"],
                    'fmeasure': metrics_result["F-mesure"],
                    'coverage_rate': metrics_result["Covered Rate"],
                    'accuracy': metrics_result["Accuracy"],
                    'mmr': metrics_result["MMR"],
                    'jaccard': metrics_result["Jaccard"],
                    'total_score': metrics_result["Score Total"]
                })
                
                print(f"\nEWCA Solution {i+1} (threshold={ss_threshold:.2f}):")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {metrics['fmeasure']:.4f}, Accuracy: {metrics['accuracy']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 1,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        
        # Method 2: SEDMTG (8 solutions)
        print("\nRunning SEDMTG method...")
        network = ProteinNetwork.from_weighted_network(sedmtg_file, has_header=True)
        sedmtg = SEDMTG(network, iterations=5)
        
        for i in tqdm(range(2), desc="SEDMTG progress"):
            # Run SEDMTG (modified to return complexes for each iteration)
            protein_complexes = sedmtg.detect_complexes()
            
            # Convert to list format and filter
            solution_complexes = [
                proteins for proteins in protein_complexes.values()
                if len(proteins) >= 3
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 3,
                'method': 'SEDMTG',
                'param': f"iteration_{i + 1}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                metrics_result = compute_metrics(solution_complexes, self.known_complexes)
                metrics.update({
                    'PPV': metrics_result["PPV"],
                    'recall': metrics_result["Recall (Sn)"],
                    'fmeasure': metrics_result["F-mesure"],
                    'coverage_rate': metrics_result["Covered Rate"],
                    'accuracy': metrics_result["Accuracy"],
                    'mmr': metrics_result["MMR"],
                    'jaccard': metrics_result["Jaccard"],
                    'total_score': metrics_result["Score Total"]
                })
                
                print(f"\nSEDMTG Solution {i+3}:")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {metrics['fmeasure']:.4f}, Accuracy: {metrics['accuracy']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 9,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        
        # Method 3: MPCC with varying filter thresholds (8 solutions)
        print("\nRunning MPCC method...")
        mpcc = MPCC()
        
        # Load interactions once (same file as EWCA)
        mpcc.load_interactions(ewca_file)
        mpcc.remove_false_positives()
        
        # Calculate topology scores and combined weights (done once)
        N = len(mpcc.id_label)
        topo_weights = mpcc.calculate_topology_scores(mpcc.relations, N)
        combined_weights = zeros((N, N))
        
        # Create weight matrix
        weight_matrix = zeros((N, N))
        for (i,j), w in mpcc.weights.items():
            weight_matrix[i,j] = w
        
        # Combine weights
        for i in range(N):
            for j in range(N):
                if i < j and weight_matrix[i,j] > 0 and topo_weights[i,j] > 0:
                    combined_weights[i,j] = 2 * weight_matrix[i,j] * topo_weights[i,j] / (weight_matrix[i,j] + topo_weights[i,j])
                    combined_weights[j,i] = combined_weights[i,j]
        
        # Detect seeds once (they don't depend on the filter threshold)
        seeds = mpcc.detect_seeds(list(mpcc.relations.keys()), mpcc.relations, combined_weights)
        
        # Vary the filter threshold from 0.1 to 0.8 in 8 steps
        filter_thresholds = np.linspace(0.9, 1.0, 2)
        
        for i, threshold in enumerate(tqdm(filter_thresholds, desc="MPCC progress")):
            # Identify complexes with current filter threshold
            complexes = mpcc.identify_complexes(seeds, mpcc.relations, combined_weights)
            
            # Calculate scores
            complex_scores = {}
            final_complexes = defaultdict(list)
            count = 1
            
            for cid in list(complexes.keys()):
                if len(complexes[cid]) >= 3:
                    final_complexes[count] = complexes[cid]
                    complex_scores[count] = mpcc.graph_entropy(complexes[cid], mpcc.relations, combined_weights)
                    count += 1
            
            # Filter with current threshold
            filtered = mpcc.filter_redundant(final_complexes, threshold=threshold)
            
            # Convert to protein names
            solution_complexes = [
                [mpcc.id_label[pid] for pid in members]
                for members in filtered.values()
            ]
            
            # Calculate metrics for this solution only
            metrics = {
                'solution_id': i + 5,  # Starts after EWCA (8) and SEDMTG (8)
                'method': 'MPCC',
                'param': f"filter_threshold={threshold:.2f}",
                'detected_complexes': len(solution_complexes),
                'known_complexes': len(self.known_complexes)
            }
            
            if self.known_complexes and solution_complexes:
                # Comparison metrics
                metrics_result = compute_metrics(solution_complexes, self.known_complexes)
                metrics.update({
                    'PPV': metrics_result["PPV"],
                    'recall': metrics_result["Recall (Sn)"],
                    'fmeasure': metrics_result["F-mesure"],
                    'coverage_rate': metrics_result["Covered Rate"],
                    'accuracy': metrics_result["Accuracy"],
                    'mmr': metrics_result["MMR"],
                    'jaccard': metrics_result["Jaccard"],
                    'total_score': metrics_result["Score Total"]
                })
                
                print(f"\nMPCC Solution {i+17} (threshold={threshold:.2f}):")
                print(f"Complexes: {len(solution_complexes)}, F-measure: {metrics['fmeasure']:.4f}, Accuracy: {metrics['accuracy']:.4f}")
            else:
                metrics.update({
                    'fmeasure': 0, 'coverage_rate': 0, 'accuracy': 0,
                    'mmr': 0, 'jaccard': 0, 'total_score': 0,
                })
                print("No known complexes - skipping metrics calculation")
            
            metrics_results.append(metrics)
            
            # Add to all complexes output
            for complex_id, proteins in enumerate(solution_complexes, 1):
                all_complexes.append({
                    'solution_id': i + 17,
                    'complex_id': complex_id,
                    'proteins': proteins
                })
        
        # Save results
        self._save_results(all_complexes, output_file)
        self._save_metrics(metrics_results, metrics_file)
        
        elapsed_time = time.time() - start_time
        print(f"\nTotal processing time: {elapsed_time:.2f} seconds")
        print(f"Complexes saved to {output_file}")
        print(f"Metrics saved to {metrics_file}")
    
    def _save_results(self, all_complexes: List[Dict], output_file: str):
        """Save all complexes to output file in the required format"""
        with open(output_file, 'w') as f:
            f.write("SolutionID\tComplexID\tProteins\n")
            for complex_data in all_complexes:
                f.write(f"{complex_data['solution_id']}\t{complex_data['complex_id']}\t{' '.join(complex_data['proteins'])}\n")
    
    def _save_metrics(self, metrics_results: List[Dict], metrics_file: str):
        """Save metrics to a TSV file with additional information"""
        with open(metrics_file, 'w') as f:
            # Write header
            f.write("SolutionID\tMethod\tParameters\tDetectedComplexes\tKnownComplexes\t"
                    "PPV\tRecall\tF-measure\tCoverageRate\tAccuracy\tMMR\tJaccard\tTotalScore\n"
                )

            # Write data
            for result in metrics_results:
                f.write(
                    f"{result['solution_id']}\t"
                    f"{result['method']}\t"
                    f"{result['param']}\t"
                    f"{result['detected_complexes']}\t"
                    f"{result['known_complexes']}\t"
                    f"{result.get('PPV', 0):.4f}\t"
                    f"{result.get('recall', 0):.4f}\t"
                    f"{result['fmeasure']:.4f}\t"
                    f"{result['coverage_rate']:.4f}\t"
                    f"{result['accuracy']:.4f}\t"
                    f"{result['mmr']:.4f}\t"
                    f"{result['jaccard']:.4f}\t"
                    f"{result['total_score']:.4f}\n"
                )

def process_multiple_files():
    # Configuration des fichiers d'entrée et de sortie
    base_dir = "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data"
    
    # Liste des fichiers d'interactions (à adapter selon vos besoins)
    interaction_files = [
        "weighted_BIOGRID_humain.txt",
        "weighted_BIOGRID_yeast.txt",
        "weighted_DIP_humain.txt",
        "weighted_DIP_yeast.txt",
        "weighted_IntAct_humain.txt"
    ]
    
    # Liste des fichiers de complexes connus correspondants
    known_complexes_files = [
        "BIOGRID_humain.txt",
        "BIOGRID_yeast.txt",
        "DIP_humain.txt",
        "DIP_yeast.txt",
        "IntAct_humain.txt"
    ]
    
    # Dossier de sortie pour les résultats
    output_base_dir = "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization"
    
    # Créer les dossiers de sortie s'ils n'existent pas
    os.makedirs(os.path.join(output_base_dir, "complexes"), exist_ok=True)
    os.makedirs(os.path.join(output_base_dir, "metrics"), exist_ok=True)
    
    # Traiter chaque fichier d'interaction
    for i, (interaction_file, known_complex_file) in enumerate(zip(interaction_files, known_complexes_files)):
        print(f"\nProcessing file {i+1}/{len(interaction_files)}: {interaction_file}")
        
        # Chemins complets des fichiers
        ewca_file = os.path.join(base_dir, "weighted_networks", interaction_file)
        sedmtg_file = os.path.join(base_dir, "weighted_networks", "tmp", f"GO_{interaction_file}")
        known_complexes_path = os.path.join(base_dir, "complexes", known_complex_file)
        
        # Noms des fichiers de sortie
        output_name = os.path.splitext(interaction_file)[0].replace("weighted_", "")
        output_file = os.path.join(output_base_dir, "complexes", f"detected_complexes_{output_name}.txt")
        metrics_file = os.path.join(output_base_dir, "metrics", f"metrics_{output_name}.tsv")
        
        # Créer l'extracteur
        extractor = ProteinComplexExtractor()
        
        # Charger les complexes connus si disponible
        if os.path.exists(known_complexes_path):
            print(f"Loading known complexes from {known_complex_file}...")
            extractor.load_known_complexes(known_complexes_path)
            print(f"Loaded {len(extractor.known_complexes)} known complexes")
        else:
            print(f"Warning: No known complexes file found at {known_complexes_path}")
        
        # Générer les complexes et métriques
        print(f"\nGenerating protein complexes and calculating metrics for {interaction_file}...")
        extractor.generate_complexes(
            ewca_file=ewca_file,
            sedmtg_file=sedmtg_file,
            output_file=output_file,
            metrics_file=metrics_file
        )
        
        print(f"\nProcessing complete for {interaction_file}!")

if __name__ == "__main__":
    process_multiple_files()


Processing file 1/5: weighted_BIOGRID_humain.txt
Loading known complexes from BIOGRID_humain.txt...
Loaded 1524 known complexes

Generating protein complexes and calculating metrics for weighted_BIOGRID_humain.txt...

Running EWCA method...


EWCA progress:   0%|          | 0/2 [00:00<?, ?it/s]

Total number of proteins: 11122
Total number of interactions: 86982


EWCA progress:   0%|          | 0/2 [00:00<?, ?it/s]


KeyboardInterrupt: 